In [ ]:
import requests
import pandas as pd
from datetime import datetime

# NASA EONET API URL (Last 365 Days)
url = "https://eonet.gsfc.nasa.gov/api/v3/events?days=365"

# Fetch data from NASA API
response = requests.get(url)

# Convert response into JSON
data = response.json()

# Extract disaster events
events = data['events']

# Empty list to store cleaned data
disaster_list = []

# Current date for calculations
current_date = datetime.now()

# Month Name Dictionary
month_names = {
    1: "January",
    2: "February",
    3: "March",
    4: "April",
    5: "May",
    6: "June",
    7: "July",
    8: "August",
    9: "September",
    10: "October",
    11: "November",
    12: "December"
}

# Loop through all disaster events
for event in events:

    try:
        # Event ID
        event_id = event['id']

        # Disaster Title
        title = event['title']

        # Disaster Category
        category = event['categories'][0]['title']

        # Clean Date (Remove Time)
        raw_date = event['geometry'][0]['date']
        clean_date = raw_date[:10]

        # Convert to datetime
        event_datetime = datetime.strptime(clean_date, "%Y-%m-%d")

        # Extract Year and Month
        year = event_datetime.year
        month = event_datetime.month
        month_name = month_names[month]

        # Source
        source = event['sources'][0]['id']

        # Coordinates
        latitude = event['geometry'][0]['coordinates'][1]
        longitude = event['geometry'][0]['coordinates'][0]

        # Event Age in Days
        event_age_days = (current_date - event_datetime).days

        # Event Age Category
        if event_age_days <= 30:
            event_age_category = "Recent"

        elif event_age_days <= 90:
            event_age_category = "Moderate"

        else:
            event_age_category = "Old"

        # Dynamic Status Logic
        if event_age_days <= 30:
            status = "Active"

        elif event_age_days <= 90:
            status = "Monitoring"

        else:
            status = "Closed"

        # Severity Classification
        if category in ['Wildfires', 'Volcanoes', 'Severe Storms']:
            severity = "High"

        elif category in ['Floods', 'Landslides']:
            severity = "Medium"

        else:
            severity = "Low"

        # Risk Score
        if severity == "High":
            risk_score = 3

        elif severity == "Medium":
            risk_score = 2

        else:
            risk_score = 1

        # Disaster Type Group
        if category in ['Wildfires', 'Severe Storms', 'Floods']:
            disaster_type_group = "Climate"

        elif category in ['Volcanoes', 'Earthquakes', 'Landslides']:
            disaster_type_group = "Geological"

        else:
            disaster_type_group = "Environmental"

        # Region Classification
        if latitude >= 15 and longitude <= -30:
            region = "North America"

        elif latitude < -60:
            region = "Antarctica"

        elif longitude >= 60 and longitude <= 150:
            region = "Asia"

        elif longitude >= 110 and latitude < 0:
            region = "Australia/Oceania"

        elif longitude >= -90 and longitude <= -30 and latitude < 15:
            region = "South America"

        else:
            region = "Other"

        # Data Collection Date
        data_collection_date = current_date.strftime("%Y-%m-%d")

        # Append cleaned data
        disaster_list.append({
            'Event ID': event_id,
            'Title': title,
            'Category': category,
            'Disaster Type Group': disaster_type_group,
            'Date': clean_date,
            'Year': year,
            'Month': month,
            'Month Name': month_name,
            'Source': source,
            'Latitude': latitude,
            'Longitude': longitude,
            'Region': region,
            'Event Age Days': event_age_days,
            'Event Age Category': event_age_category,
            'Status': status,
            'Severity': severity,
            'Risk Score': risk_score,
            'Data Collection Date': data_collection_date
        })

    except:
        pass

# Convert list into DataFrame
df = pd.DataFrame(disaster_list)

# Keep only recent data (2024 onwards)
df = df[df['Year'] >= 2024]

# Remove duplicate records
df.drop_duplicates(inplace=True)

# Reset index
df.reset_index(drop=True, inplace=True)

# Display dataset
print(df)

# Display disaster category counts
print("\nDisaster Category Counts:\n")
print(df['Category'].value_counts())

# Display region counts
print("\nRegion Counts:\n")
print(df['Region'].value_counts())

# Save final cleaned dataset as CSV
df.to_csv("disaster_data.csv", index=False)

print("\nFinal CSV file created successfully!")

         Event ID                                      Title  \
0     EONET_24786                        Tropical Storm Nolo   
1     EONET_24787                     Tropical Storm Surigae   
2     EONET_24809  Wildfire Merit Creek, Greene, Mississippi   
3     EONET_24785                       Tropical Cyclone 01B   
4     EONET_24721                             Hurricane Polo   
...           ...                                        ...   
1583  EONET_15680         Lake Creek Wildfire, Blaine, Idaho   
1584  EONET_15626            Hayes Wildfire, Blaine, Montana   
1585   EONET_6474                                Iceberg D35   
1586   EONET_6523                                Iceberg A83   
1587  EONET_12716                                Iceberg A85   

              Category Disaster Type Group        Date  Year  Month  \
0        Severe Storms             Climate  2026-09-24  2026      9   
1        Severe Storms             Climate  2026-09-23  2026      9   
2            Wildf